# 토크나이저 4종 비교 실습

**자연어정보분석 | 연세대학교 글로벌인재대학**

하나의 문장을 4가지 토크나이저로 분석하고 결과를 비교합니다:

| # | 방식 | 라이브러리 | 원리 |
|---|------|-----------|------|
| 1 | **형태소 분석** | Kiwi | 언어학 규칙 기반 — 품사 단위 분리 |
| 2 | **BPE** | tiktoken (GPT-4) | 바이트 쌍 빈도 통계 — 바이트 단위 병합 |
| 3 | **WordPiece** | transformers (BERT) | BPE 변형 — likelihood 기반 병합, `##` 접두사 |
| 4 | **SentencePiece** | sentencepiece (T5) | 언어 무관 — 원시 텍스트에서 직접 학습, `▁` 공백 표시 |

In [ ]:
# 환경 설정
!pip install -q tiktoken kiwipiepy transformers sentencepiece protobuf

In [ ]:
import tiktoken
from kiwipiepy import Kiwi
from transformers import AutoTokenizer
import sentencepiece as spm
import os, urllib.request, tempfile

# --- 토크나이저 초기화 ---
kw = Kiwi()                                          # 형태소
bpe_enc = tiktoken.encoding_for_model("gpt-4")       # BPE (GPT-4)
wp_tok = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")  # WordPiece (mBERT)

# SentencePiece — T5 모델 사용
sp_tok = AutoTokenizer.from_pretrained("google/mt5-small")  # mT5 SentencePiece

print("✓ 4종 토크나이저 로드 완료")

## 1. 단일 문장 4종 비교

In [ ]:
def safe_decode_bpe(token_id):
    """BPE 토큰을 안전하게 디코딩 (불완전 UTF-8 → hex)"""
    raw = bpe_enc.decode_single_token_bytes(token_id)
    try:
        return raw.decode("utf-8")
    except UnicodeDecodeError:
        return f"0x{raw.hex()}"

def compare_tokenizers(text):
    """하나의 문장을 4종 토크나이저로 분석"""
    print(f'원문: "{text}"')
    print("=" * 70)

    # 1) 형태소 (Kiwi)
    kiwi_tokens = kw.tokenize(text)
    kiwi_forms = [f"{t.form}/{t.tag}" for t in kiwi_tokens]
    kiwi_plain = [t.form for t in kiwi_tokens]
    print(f"\n{'형태소 (Kiwi)':>18s} | {len(kiwi_plain):>2d}개 | {kiwi_plain}")
    print(f"{'':>18s} |      품사: {kiwi_forms}")

    # 2) BPE (GPT-4 tiktoken)
    bpe_ids = bpe_enc.encode(text)
    bpe_tokens = [safe_decode_bpe(t) for t in bpe_ids]
    print(f"\n{'BPE (GPT-4)':>18s} | {len(bpe_tokens):>2d}개 | {bpe_tokens}")

    # 3) WordPiece (mBERT)
    wp_result = wp_tok.tokenize(text)
    print(f"\n{'WordPiece (BERT)':>18s} | {len(wp_result):>2d}개 | {wp_result}")

    # 4) SentencePiece (mT5)
    sp_result = sp_tok.tokenize(text)
    print(f"\n{'SentencePiece (T5)':>18s} | {len(sp_result):>2d}개 | {sp_result}")

    print()
    return {
        "형태소": len(kiwi_plain),
        "BPE": len(bpe_tokens),
        "WordPiece": len(wp_result),
        "SentencePiece": len(sp_result),
    }

# --- 비교 실행 ---
counts = compare_tokenizers("인공지능이 자연어 처리 기술을 빠르게 발전시키고 있다")

### 관찰 포인트

- **형태소**: `인공지능/NNG` + `이/JKS` — 언어학적 의미 단위
- **BPE**: 바이트 단위로 쪼개서 한국어 글자가 2~3토큰으로 분해될 수 있음
- **WordPiece**: `##`이 붙은 토큰은 단어 중간 조각 (예: `##능`, `##이`)
- **SentencePiece**: `▁`(U+2581)은 공백 위치 표시 — 원문 복원 가능

## 2. 다양한 문장으로 비교

In [ ]:
test_sentences = [
    "Natural language processing is amazing",
    "자연어 처리는 놀랍습니다",
    "AI가 세상을 바꾸고 있다 🤖🌍",
    "Transformer 모델은 self-attention을 사용한다",
    "ㅋㅋㅋ 대박 ㅎㅎ",
    "E = mc² → 에너지 보존",
]

all_counts = []
for sent in test_sentences:
    c = compare_tokenizers(sent)
    c["문장"] = sent
    all_counts.append(c)
    print("-" * 70)

## 3. 토큰 수 비교 시각화

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = ['AppleGothic', 'Malgun Gothic', 'NanumGothic', 'sans-serif']
matplotlib.rcParams['axes.unicode_minus'] = False

methods = ["형태소", "BPE", "WordPiece", "SentencePiece"]
colors = ["#1a4e7a", "#e8862a", "#2ca02c", "#d62728"]
labels = [s[:18] + "..." if len(s) > 18 else s for s in [c["문장"] for c in all_counts]]

fig, ax = plt.subplots(figsize=(14, 6))
x = range(len(all_counts))
width = 0.2

for i, (method, color) in enumerate(zip(methods, colors)):
    values = [c[method] for c in all_counts]
    bars = ax.bar([xi + width * i for xi in x], values, width,
                  label=method, color=color, edgecolor="white")
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                str(int(bar.get_height())), ha="center", fontsize=8, fontweight="bold")

ax.set_xticks([xi + width * 1.5 for xi in x])
ax.set_xticklabels(labels, fontsize=9, rotation=20, ha="right")
ax.set_ylabel("토큰 수")
ax.set_title("토크나이저 4종 비교 — 같은 문장의 토큰 수", fontweight="bold", fontsize=13)
ax.legend(loc="upper left")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig("images/w02-tokenizer-comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("→ images/w02-tokenizer-comparison.png 저장됨")

## 4. 토큰 분해 상세 비교 — 한 글자씩 추적

In [ ]:
def detailed_comparison(text):
    """한 문장의 토큰 분해를 표 형태로 비교"""
    print(f'\n{"="*70}')
    print(f'  "{text}"')
    print(f'{"="*70}\n')

    # 각 토크나이저 결과
    kiwi_res = [f"{t.form}/{t.tag}" for t in kw.tokenize(text)]
    bpe_ids = bpe_enc.encode(text)
    bpe_res = [safe_decode_bpe(t) for t in bpe_ids]
    wp_res = wp_tok.tokenize(text)
    sp_res = sp_tok.tokenize(text)

    rows = [
        ("형태소 (Kiwi)",      kiwi_res),
        ("BPE (GPT-4)",        bpe_res),
        ("WordPiece (BERT)",   wp_res),
        ("SentencePiece (T5)", sp_res),
    ]

    for name, tokens in rows:
        token_str = " | ".join(tokens)
        print(f"  {name:>20s} ({len(tokens):>2d}개)")
        print(f"  {'':>20s}  [{token_str}]\n")

    # 복원 테스트
    print("  --- 원문 복원 테스트 ---")
    bpe_restored = bpe_enc.decode(bpe_ids)
    wp_restored = wp_tok.convert_tokens_to_string(wp_res)
    sp_restored = sp_tok.convert_tokens_to_string(sp_res)
    kiwi_joined = "".join(t.form for t in kw.tokenize(text))

    for name, restored in [("BPE", bpe_restored), ("WordPiece", wp_restored),
                            ("SentencePiece", sp_restored), ("형태소(join)", kiwi_joined)]:
        match = "✓" if restored.strip() == text.strip() else "✗"
        print(f"  {name:>16s}: {match} \"{restored}\"")

detailed_comparison("인공지능이 자연어 처리 기술을 빠르게 발전시키고 있다")
detailed_comparison("Transformer uses self-attention mechanism")
detailed_comparison("GPT-4는 BPE, BERT는 WordPiece를 사용한다")

## 5. 알고리즘별 핵심 차이 정리

| 특성 | 형태소 분석 | BPE | WordPiece | SentencePiece |
|------|------------|-----|-----------|---------------|
| **분리 기준** | 언어학 규칙 | 바이트 쌍 빈도 | likelihood 최대화 | unigram 확률 |
| **사전 학습** | 언어별 사전 필요 | 말뭉치에서 학습 | 말뭉치에서 학습 | 말뭉치에서 학습 |
| **다국어** | 언어별 도구 필요 | 바이트 기반 → 모든 언어 | 유니코드 기반 | 원시 텍스트 → 모든 언어 |
| **OOV 처리** | 미등록어 문제 | 바이트까지 분해 → OOV 없음 | `[UNK]` 가능 | 문자까지 분해 → OOV 없음 |
| **공백 처리** | 공백 제거 | 공백을 토큰에 포함 | 공백에서 분리 + `##` | `▁`로 공백 위치 표시 |
| **대표 모델** | KoNLPy, Kiwi, MeCab | GPT-2/3/4 | BERT, DistilBERT | T5, mT5, ALBERT, XLNet |
| **한국어 효율** | 높음 (의미 단위) | 낮음 (바이트 분해) | 중간 | 중간~높음 |

## 6. 한국어 효율 실험 — 같은 의미, 토큰 수 비교

In [ ]:
pairs = [
    ("Artificial intelligence", "인공지능"),
    ("Natural language processing", "자연어 처리"),
    ("Machine learning model", "기계학습 모델"),
    ("Deep learning is powerful", "딥러닝은 강력하다"),
    ("The cat sat on the mat", "고양이가 매트 위에 앉았다"),
]

print(f'{"문장":^30s} {"BPE":>5s} {"WP":>5s} {"SP":>5s} {"형태소":>6s}')
print("-" * 60)

efficiency_data = []
for en, ko in pairs:
    for lang, text in [("EN", en), ("KO", ko)]:
        b = len(bpe_enc.encode(text))
        w = len(wp_tok.tokenize(text))
        s = len(sp_tok.tokenize(text))
        m = len(kw.tokenize(text))
        tag = f"[{lang}]"
        print(f'{tag + " " + text:30s} {b:>5d} {w:>5d} {s:>5d} {m:>6d}')
        efficiency_data.append({"lang": lang, "text": text,
                                 "BPE": b, "WordPiece": w, "SentencePiece": s, "형태소": m})
    print()

# 영어 vs 한국어 효율 비율
print("\n=== 한국어/영어 토큰 수 비율 (>1이면 한국어가 더 많은 토큰 필요) ===")
for i in range(0, len(efficiency_data), 2):
    en_d = efficiency_data[i]
    ko_d = efficiency_data[i + 1]
    ratios = {m: ko_d[m] / en_d[m] for m in ["BPE", "WordPiece", "SentencePiece", "형태소"]}
    print(f'  {ko_d["text"]:20s}  BPE {ratios["BPE"]:.1f}x  WP {ratios["WordPiece"]:.1f}x  '
          f'SP {ratios["SentencePiece"]:.1f}x  형태소 {ratios["형태소"]:.1f}x')

## 7. 도전 과제

1. **나만의 문장 실험**: 아래 셀에 원하는 문장을 넣고 4종 비교를 해보세요
2. **코드 토큰화**: `print("hello world")` 같은 코드를 넣으면 어떤 토크나이저가 효율적인가?
3. **긴 문장 vs 짧은 문장**: 문장 길이에 따라 토크나이저별 효율이 어떻게 달라지는지 관찰

In [ ]:
# 여기에 원하는 문장을 넣어보세요!
my_text = "여기에 원하는 문장을 입력하세요"

compare_tokenizers(my_text)
detailed_comparison(my_text)